In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from statistical_analysis_and_plots import get_data, bootstrap_confidence_interval

from tqdm.auto import tqdm
import torch

/gpfs/data/chopralab/singhr36/miniconda/envs/d1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dict_mc_scores, dict_judge_scores, dict_importance_weights, prompt_category_index, all_outputs, _ = get_data(
        "output_paraphrases_est_unsafe_to_unsafe/", 
    save_cheap_model_output=False, 
    min_sample_size=1000, 
    model_name_to_load='phi',
    )

Model: phi-4, Experiment: 20260118_121610
Skipping phi-4_500_42 because it has less than 1000 samples.
Model: phi-4, Experiment: 20260118_184706
Not outptutting model responses and completion ids to save memory since we only need the scores for analysis.
Including Limit Samples: [5000, 6250]
MC Scores Shape: torch.Size([1250, 10000]), IS Scores Shape: torch.Size([1250, 500])
CEM True, Ablation 1.0, Num Particles 500, Seed 44
MC Mean:  tensor(0.0027, dtype=torch.float64)
-----


Model: phi-4, Experiment: 20260118_121533
Skipping phi-4_500_42 because it has less than 1000 samples.
Model: phi-4, Experiment: 20260118_121723
Skipping phi-4_500_42 because it has less than 1000 samples.
Model: phi-4, Experiment: 20260118_161517
Not outptutting model responses and completion ids to save memory since we only need the scores for analysis.
Including Limit Samples: [1250, 2500]
MC Scores Shape: torch.Size([1250, 10000]), IS Scores Shape: torch.Size([1250, 500])
CEM True, Ablation 1.0, Num Particle

In [ ]:
dict_judge_scores.keys()

In [ ]:
all_para_is_grouped = torch.zeros(309, 25)
for key in dict_judge_scores:
    all_judge = dict_judge_scores[key]
    all_iw = dict_importance_weights[key]

    all_is = (dict_judge_scores[key] > 0.75).float() *  dict_importance_weights[key].squeeze(2)
    
    n_prompts = all_is.shape[0] // 25

    
    limit_samples = [int(x) // 25 for x in key.split('limit')[1].split('_')[1:]]    
    para_is_grouped = torch.cat([all_is[25 * idx: 25 * (idx + 1)].unsqueeze(0) for idx in range(n_prompts)], dim=0).mean(dim=2)    
    all_para_is_grouped[limit_samples[0]: limit_samples[1]] = para_is_grouped

    print(key, para_is_grouped.shape)
    
all_para_is_grouped.shape

In [ ]:
prompt_category_index.keys(), [len(prompt_category_index[key]) for key in prompt_category_index]

In [ ]:
indices = prompt_category_index['Hate, harassment and discrimination']

In [ ]:
# def empirical_estimator_for_max_prob(data, idx, prob):
#     n_trial = 500
#     val = sum([permute(data)[:idx].max() > prob for _ in range(n_trial)])/n_trial

#     return val
# for key in prompt_category_index.keys():
#     indices = prompt_category_index[key]

#     print(all_para_is_grouped[indices].shape, key)
#     train_data = all_para_is_grouped[indices, :15].flatten()
#     cdf_estimator = empirical_cdf(train_data)
    
#     plt.plot(np.linspace(0, 1, 100), [cdf_estimator(t) for t in np.linspace(0, 1, 100)])
#     plt.show()

In [ ]:
all_para_is_grouped[indices].shape

In [ ]:
torch.sort(all_para_is_grouped[indices, 1:].mean(dim=0))

In [ ]:
def empirical_cdf(x):
    x = np.sort(np.asarray(x))
    n = len(x)

    def cdf(t):
        return np.searchsorted(x, t, side='right') / n

    return cdf


In [ ]:
def permute(x):
    n = len(x)
    random_indices = torch.randperm(n)
    return x[random_indices]
permute(torch.arange(10))

In [ ]:
torch.cat([torch.ones(10).unsqueeze(0)*_ for _ in range(2)], dim=0).shape, torch.cat([torch.ones(10).unsqueeze(0)*_ for _ in range(2)], dim=0).flatten()

In [ ]:
all_para_is_grouped[indices].shape

In [ ]:
# train_data = all_para_is_grouped[indices, :10].flatten()
# test_data = all_para_is_grouped[indices, 10:].flatten()

# all_data = all_para_is_grouped.flatten()

# train_data = all_para_is_grouped[:150, :].flatten()
# test_data = all_para_is_grouped[150:, :].flatten()

all_data = all_para_is_grouped.flatten()
all_data = permute(all_data)

n_train = int(len(all_data) * 0.4)
train_data = all_data[:n_train]
test_data = all_data[n_train:]

train_data.shape, test_data.shape, all_data.shape

In [ ]:
def empirical_estimator_for_max_prob(data, n_queries, prob):
    n_trial = 800
    val = sum([permute(data)[:n_queries].max() > prob for _ in range(n_trial)])/n_trial

    return val

cdf_estimator = empirical_cdf(train_data)

plt.plot(np.linspace(0, 1, 100), [cdf_estimator(t) for t in np.linspace(0, 1, 100)])

In [ ]:
cdf_estimator = empirical_cdf(train_data)
prob = 0.9
n_queries = range(5, 1500, 50)

fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=100)

test_val = [empirical_estimator_for_max_prob(test_data, n_query, prob=prob) for n_query in n_queries]
predicted_val = [1-cdf_estimator(prob)**n_query for n_query in n_queries]

ax.plot(n_queries, test_val, label='Test')
ax.plot(n_queries, predicted_val, label='Predicted')

ax.set_xlabel('N Queries')
ax.set_ylabel(r'P$(\text{max}_{i \leq n} h_i >$' + '$ {})$'.format(prob))

ax.grid(True, c='0.9', axis='y')
ax.grid(True, c='0.9', axis='x')


ax.legend()

ax.tick_params(axis='both', which='major', labelsize=15)
ax.tick_params(axis='both', which='minor', labelsize=15)

In [ ]:
test_data.mean(), train_data.mean()